In [25]:
print("Setup working")

Setup working


In [26]:
import os
os.listdir("../data/raw/")

['assessments.csv',
 'courses.csv',
 'studentAssessment.csv',
 'studentInfo.csv',
 'studentRegistration.csv',
 'studentVle.csv',
 'vle.csv']

In [27]:
import pandas as pd

path = "../data/raw/"

student_info = pd.read_csv(path + "studentInfo.csv")
student_vle = pd.read_csv(path + "studentVle.csv")
assessments = pd.read_csv(path + "assessments.csv")
student_assessment = pd.read_csv(path + "studentAssessment.csv")
courses = pd.read_csv(path + "courses.csv")

In [28]:
student_info.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [29]:
student_vle.head()

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [30]:
student_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [31]:
student_info.shape
student_info.columns
student_info.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

In [32]:
student_vle.shape
student_vle.columns
student_vle.isnull().sum()

code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

In [33]:
student_vle.shape
student_vle.columns
student_vle.isnull().sum()

code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

In [34]:
student_assessment.shape
student_assessment.columns
student_assessment.isnull().sum()

id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

In [35]:
assessments.shape
assessments.columns
assessments.isnull().sum()

code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

In [36]:
courses.shape
courses.columns
courses.isnull().sum()

code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64

In [37]:
# Convert to week
student_vle['week'] = (student_vle['date'] // 7) + 1
student_assessment['week'] = (student_assessment['date_submitted'] // 7) + 1

# Weekly clicks
weekly_clicks = student_vle.groupby(['id_student', 'week'])['sum_click'].sum().reset_index()

# Weekly scores
weekly_scores = student_assessment.groupby(['id_student', 'week'])['score'].mean().reset_index()

# Merge
df = weekly_clicks.merge(weekly_scores, on=['id_student', 'week'], how='left')
df = df.merge(student_info[['id_student', 'final_result']], on='id_student', how='left')

# Fill missing
df['score'] = df['score'].fillna(0)

In [38]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[['click_scaled', 'score_scaled']] = scaler.fit_transform(df[['sum_click', 'score']])

df['engagement_score'] = 0.7 * df['click_scaled'] + 0.3 * df['score_scaled']

In [39]:
df['label'] = df['final_result'].apply(lambda x: 1 if x in ['Fail', 'Withdraw'] else 0)

In [40]:
df_model = df[df['week'] <= 6]

df_model = df_model.groupby('id_student').agg({
    'sum_click': 'sum',
    'score': 'mean',
    'label': 'max'
}).reset_index()

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X = df_model[['sum_click', 'score']]
y = df_model['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       0.82      0.37      0.51      3932
           1       0.27      0.75      0.40      1241

    accuracy                           0.46      5173
   macro avg       0.55      0.56      0.46      5173
weighted avg       0.69      0.46      0.49      5173

ROC-AUC: 0.5849956922804518


In [42]:
# Average score per course (proxy recommendation)
course_perf = student_assessment.merge(assessments, on='id_assessment')

course_scores = course_perf.groupby('code_module')['score'].mean().sort_values(ascending=False)

top_courses = course_scores.head(3)
print(top_courses)

code_module
EEE    81.180066
GGG    79.700493
FFF    77.707590
Name: score, dtype: float64


In [43]:
# Merge course info with scores
course_perf = student_assessment.merge(assessments, on='id_assessment')

# Average score per course (proxy for "good courses")
course_scores = course_perf.groupby('code_module')['score'].mean().sort_values(ascending=False)

# Top 3 recommendations
top_courses = course_scores.head(3)

print("Top Recommended Courses:")
print(top_courses)

Top Recommended Courses:
code_module
EEE    81.180066
GGG    79.700493
FFF    77.707590
Name: score, dtype: float64
